In [1]:
# 1. Setup Repo and Libraries
!git clone https://github.com/mlpc-ucsd/Patch-DM.git
%cd Patch-DM
!pip install -q pytorch-lightning==1.9.5 lmdb==1.2.1 lpips pytorch-fid einops
!pip install -q git+https://github.com/openai/CLIP.git

import os
import re
import torch

# 2. Patch Code for Kaggle Compatibility
os.system('git checkout experiment.py train.py')

# Fix experiment.py (Numpy, Hooks, and Trainer Timeouts)
with open('experiment.py', 'r') as f:
    content = f.read()
content = content.replace('from numpy.lib.function_base import flip', 'from numpy import flip')
content = re.sub(
    r'def on_train_batch_end\s*\([^)]+\)\s*->\s*None:',
    r'def on_train_batch_end(self, outputs, batch, batch_idx: int) -> None:',
    content
)
content = content.replace("desc='infer')", "desc='infer', disable=True)")
content = content.replace(
    'trainer = pl.Trainer(', 
    'trainer = pl.Trainer(enable_progress_bar=False, max_time="00:11:30:00", '
)
with open('experiment.py', 'w') as f:
    f.write(content)

# Fix train.py (Checkpoint Frequency)
with open('train.py', 'r') as f:
    train_content = f.read()
train_content = train_content.replace('conf.save_every_samples = 100_000', 'conf.save_every_samples = 10_000')
with open('train.py', 'w') as f:
    f.write(train_content)

print("✅ Environment ready and code patched.")

Cloning into 'Patch-DM'...
remote: Enumerating objects: 93, done.
remote: Counting objects: 100% (93/93), done.
remote: Compressing objects: 100% (75/75), done.
remote: Total 93 (delta 21), reused 85 (delta 16), pack-reused 0 (from 0)
Receiving objects: 100% (93/93), 1.06 MiB | 9.37 MiB/s, done.
Resolving deltas: 100% (21/21), done.
/kaggle/working/Patch-DM
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 881.5/881.5 kB 15.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 829.5/829.5 kB 45.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.8/53.8 kB 3.8 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 1.7 MB/s eta 0:00:00
✅ Environment ready and code patched.


Updated 0 paths from the index


In [2]:
# Paths based on your screenshot
INPUT_SEMANTIC = "/kaggle/input/datasets/vanditaguptabest/lmdb-input/lmdb-input/semantic.pt"
FIXED_SEMANTIC = "/kaggle/working/semantic_fixed.pt"

if os.path.exists(INPUT_SEMANTIC):
    data = torch.load(INPUT_SEMANTIC, map_location='cpu')
    
    # Check if it needs wrapping into the 'model_state_dict' envelope
    if torch.is_tensor(data):
        print("📦 Input is a raw tensor. Wrapping for Patch-DM...")
        wrapped = {'model_state_dict': {'semantic_enc.weight': data}}
        torch.save(wrapped, FIXED_SEMANTIC)
    elif isinstance(data, dict) and 'model_state_dict' in data:
        print("✅ Input is already in the correct dictionary format.")
        torch.save(data, FIXED_SEMANTIC)
    else:
        # If it's a dict but has a different key
        print("🔧 Re-keying dictionary for training...")
        key = list(data.keys())[0] if isinstance(data, dict) else None
        tensor = data[key] if key else data
        wrapped = {'model_state_dict': {'semantic_enc.weight': tensor}}
        torch.save(wrapped, FIXED_SEMANTIC)
        
    print(f"🚀 Semantic file ready at: {FIXED_SEMANTIC}")
else:
    print("❌ Could not find semantic.pt in Input. Check dataset attachment!")

✅ Input is already in the correct dictionary format.
🚀 Semantic file ready at: /kaggle/working/semantic_fixed.pt


In [3]:
import subprocess
import time
import os

# 1. SMART PATH FINDER: Finds where train.py is hiding
target_file = 'train.py'
found_path = None
for root, dirs, files in os.walk('/kaggle/working'):
    if target_file in files:
        found_path = root
        break

if found_path:
    print(f"📂 Found train.py in: {found_path}")
    os.chdir(found_path)
else:
    raise FileNotFoundError("❌ Could not find train.py! Check your git clone step.")

# 2. Paths
DATA_PATH = "/kaggle/input/datasets/vanditaguptabest/lmdb-input/lmdb-input/ffhq256.lmdb"
SEMANTIC_PATH = "/kaggle/working/semantic_fixed.pt"
CHECKPOINT_DIR = os.path.join(found_path, 'checkpoints/patchdm_ffhq/')

# 3. Start Training (Using -u for real-time logs)
print("🔥 Launching Training (Unbuffered Logs)...")
train_cmd = [
    "python", "-u", "train.py",
    "--batch_size", "4",
    "--patch_size", "64",
    "--data_path", DATA_PATH,
    "--name", "patchdm_ffhq",
    "--semantic_path", SEMANTIC_PATH
]

# Redirect stdout and stderr to the notebook's output
process = subprocess.Popen(train_cmd, stdout=None, stderr=None)

# 4. Disk Guardian
print("🛡️ Disk Guardian Active...")
try:
    while process.poll() is None:
        if os.path.exists(CHECKPOINT_DIR):
            files = [os.path.join(CHECKPOINT_DIR, f) for f in os.listdir(CHECKPOINT_DIR) if f.endswith('.ckpt')]
            if len(files) > 2:
                files.sort(key=os.path.getmtime)
                for f in files[:-2]: 
                    os.remove(f)
                    print(f"🗑️ Cleaned old checkpoint: {os.path.basename(f)}")
        time.sleep(600) 
    
    if process.returncode != 0:
        print(f"❌ Training stopped unexpectedly with code {process.returncode}")
except KeyboardInterrupt:
    process.terminate()

# 5. Final Backup
print("📦 Zipping results...")
%cd /kaggle/working/
!zip -r /kaggle/working/final_weights.zip {CHECKPOINT_DIR}

📂 Found train.py in: /kaggle/working/Patch-DM
🔥 Launching Training (Unbuffered Logs)...
🛡️ Disk Guardian Active...
conf: patchdm_ffhq


Global seed set to 0


Model params: 102.28 M
==Model size is 64==
ckpt path: checkpoints/patchdm_ffhq/last.ckpt


ModelCheckpoint(save_last=True, save_top_k=-1, monitor=None) will duplicate the last checkpoint saved.
/usr/local/lib/python3.12/dist-packages/pytorch_lightning/trainer/connectors/accelerator_connector.py:478: LightningDeprecationWarning: Setting `Trainer(gpus=[0])` is deprecated in v1.7 and will be removed in v2.0. Please use `Trainer(accelerator='gpu', devices=[0])` instead.
  rank_zero_deprecation(
Using 16bit None Automatic Mixed Precision (AMP)
/usr/local/lib/python3.12/dist-packages/pytorch_lightning/plugins/precision/native_amp.py:47: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler()
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
IPU available: False, using: 0 IPUs
HPU available: False, using: 0 HPUs


local seed: 0
train data: 70000
val data: 70000


...

🗑️ Cleaned old checkpoint: epoch=0-step=2500.ckpt
🗑️ Cleaned old checkpoint: epoch=0-step=5000.ckpt
🗑️ Cleaned old checkpoint: epoch=0-step=7500.ckpt
🗑️ Cleaned old checkpoint: epoch=0-step=10000.ckpt
🗑️ Cleaned old checkpoint: epoch=0-step=12500.ckpt
🗑️ Cleaned old checkpoint: epoch=0-step=15000.ckpt
🗑️ Cleaned old checkpoint: epoch=0-step=17500.ckpt
🗑️ Cleaned old checkpoint: epoch=1-step=20000.ckpt
🗑️ Cleaned old checkpoint: epoch=1-step=22500.ckpt
🗑️ Cleaned old checkpoint: epoch=1-step=25000.ckpt
🗑️ Cleaned old checkpoint: epoch=1-step=27500.ckpt
🗑️ Cleaned old checkpoint: epoch=1-step=30000.ckpt
🗑️ Cleaned old checkpoint: epoch=1-step=32500.ckpt
🗑️ Cleaned old checkpoint: epoch=1-step=35000.ckpt
🗑️ Cleaned old checkpoint: epoch=2-step=37500.ckpt
🗑️ Cleaned old checkpoint: epoch=2-step=40000.ckpt
🗑️ Cleaned old checkpoint: epoch=2-step=42500.ckpt
🗑️ Cleaned old checkpoint: epoch=2-step=45000.ckpt
🗑️ Cleaned old checkpoint: epoch=2-step=47500.ckpt
🗑️ Cleaned old checkpoint: epoch=2

/kaggle/working/Patch-DM/diffusion/base.py:446: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(self.conf.fp16):


🗑️ Cleaned old checkpoint: epoch=6-step=122500.ckpt
🗑️ Cleaned old checkpoint: epoch=7-step=125000.ckpt
🗑️ Cleaned old checkpoint: epoch=7-step=127500.ckpt
🗑️ Cleaned old checkpoint: epoch=7-step=130000.ckpt
🗑️ Cleaned old checkpoint: epoch=7-step=132500.ckpt


Time limit reached. Elapsed time is 11:30:00. Signaling Trainer to stop.


📦 Zipping results...
/kaggle/working
  adding: kaggle/working/Patch-DM/checkpoints/patchdm_ffhq/ (stored 0%)
  adding: kaggle/working/Patch-DM/checkpoints/patchdm_ffhq/sample/ (stored 0%)
  adding: kaggle/working/Patch-DM/checkpoints/patchdm_ffhq/sample/500000.png (deflated 0%)
  adding: kaggle/working/Patch-DM/checkpoints/patchdm_ffhq/epoch=7-step=135000.ckpt (deflated 14%)
  adding: kaggle/working/Patch-DM/checkpoints/patchdm_ffhq/events.out.tfevents.1774478325.603aa85044dc.109.0 (deflated 61%)
  adding: kaggle/working/Patch-DM/checkpoints/patchdm_ffhq/last.ckpt (deflated 14%)
  adding: kaggle/working/Patch-DM/checkpoints/patchdm_ffhq/hparams.yaml (deflated 65%)


In [4]:
# STANDALONE SAFETY NET
import os
import time

# Wait 5 seconds to let any background processes settle
time.sleep(5)

# Zip everything from the root /kaggle/working/
# This captures the repo, the weights, and the semantic file
!zip -r /kaggle/working/patchdm_final_backup.zip /kaggle/working/Patch-DM/ /kaggle/working/semantic.pt

print("✅ Backup complete. Look for 'patchdm_final_backup.zip' tomorrow morning.")

	zip warning: name not matched: /kaggle/working/semantic.pt
  adding: kaggle/working/Patch-DM/ (stored 0%)
  adding: kaggle/working/Patch-DM/utils/ (stored 0%)
  adding: kaggle/working/Patch-DM/utils/align.py (deflated 68%)
  adding: kaggle/working/Patch-DM/utils/metrics.py (deflated 75%)
  adding: kaggle/working/Patch-DM/utils/renderer.py (deflated 70%)
  adding: kaggle/working/Patch-DM/utils/choices.py (deflated 71%)
  adding: kaggle/working/Patch-DM/utils/dataset_util.py (deflated 50%)
  adding: kaggle/working/Patch-DM/utils/lmdb_writer.py (deflated 66%)
  adding: kaggle/working/Patch-DM/utils/dataset.py (deflated 88%)
  adding: kaggle/working/Patch-DM/utils/dist_utils.py (deflated 70%)
  adding: kaggle/working/Patch-DM/utils/__pycache__/ (stored 0%)
  adding: kaggle/working/Patch-DM/utils/__pycache__/renderer.cpython-312.pyc (deflated 44%)
  adding: kaggle/working/Patch-DM/utils/__pycache__/lmdb_writer.cpython-312.pyc (deflated 51%)
  adding: kaggle/working/Patch-DM/utils/__pycache